# Import libraries

In [1]:
import os
import sys

project_root = "Incremental-Learning-MycroftMinds"
cwd = os.path.abspath(os.getcwd())
while True:
    if os.path.basename(cwd) == project_root:
        if cwd not in sys.path:
            sys.path.insert(0, cwd)
        break
    parent = os.path.dirname(cwd)
    if parent == cwd:
        raise FileNotFoundError(f"Project root '{project_root}' not found in any parent directory.")
    cwd = parent

In [2]:
from src.data import dataloader
from src.data.io import load_dataset, save_dataset
from src.utils.memory import print_memory_stats
from src.trees import fastodt
import numpy as np
import pandas as pd
import time

# `old_batch` splits

In [ ]:
data_dir = '../data'

lag_args = {
    #'lag_5m':'5M',
    'lag_1h':'1H',
    'lag_24h':'24H',
    'lag_168h':'168H'} # 24*7 = 168h, one week ago

# list feature names
columns_to_add_lag = [
            'total_power_active', 'total_power_reactive', 'total_power_apparent',
            'import_energy_active', 'import_energy_reactive', 'export_energy_reactive',
            'total_energy_active', 'total_energy_reactive'
        ]

# target
target_feature = 'total_power_active'

# list of devices to be used
device_list = [
    "f90e7387-9810-408d-885b-6f3841f4ec3f",
    "dd40019b-5b35-4fba-b0dd-16083c1fdb84",
    "3617dc75-38d4-4875-b66c-64f896b5c9f1",
    "a4b80cd7-5438-44cd-8257-c9ac81684822",
    "522e56a5-ff0c-4ece-923f-a925aabe15e6",
    "4a448440-d2dd-4b27-ae02-87e7e3baa9f5",
    "fca96b30-ca2f-41b8-b987-3207af296b03",
    "4ad3e3e6-5f11-4923-a88d-668b4ddb493a",
    ]

In [ ]:
# Start timer to measure processing time
start_time = time.time()

# Generate data with optimized memory usage
train, valid, test = dataloader.get_train_test_sets(
    data_dir=data_dir,
    lag_freqs=lag_args,
    columns_to_add_lag=columns_to_add_lag,
    target_feature=target_feature,
    device_list=device_list,
    walk_forward_sets=True,
    train_days=28,
    valid_days=28,
    optimize_memory=True,  # Enable memory optimization
    save=True,  # Save processed data
    intermediate_prefix='8_dev_hour_wfv',  # Prefix for saved files
    time_agg='1H',  # Time aggregation frequency
)

# Calculate and print the processing time
processing_time = time.time() - start_time
print(f"Data processing completed in {(processing_time/60):.2f} minutes")

[INFO] Aggregating data by: 1H
input_features:  ['device_id', 'time', 'export_energy_active', 'hour', 'weekday', 'month', 'total_power_active_lag_1h', 'total_power_active_lag_24h', 'total_power_active_lag_168h', 'total_power_reactive_lag_1h', 'total_power_reactive_lag_24h', 'total_power_reactive_lag_168h', 'total_power_apparent_lag_1h', 'total_power_apparent_lag_24h', 'total_power_apparent_lag_168h', 'import_energy_active_lag_1h', 'import_energy_active_lag_24h', 'import_energy_active_lag_168h', 'import_energy_reactive_lag_1h', 'import_energy_reactive_lag_24h', 'import_energy_reactive_lag_168h', 'export_energy_reactive_lag_1h', 'export_energy_reactive_lag_24h', 'export_energy_reactive_lag_168h', 'total_energy_active_lag_1h', 'total_energy_active_lag_24h', 'total_energy_active_lag_168h', 'total_energy_reactive_lag_1h', 'total_energy_reactive_lag_24h', 'total_energy_reactive_lag_168h']

input_features:  ['device_id', 'time', 'export_energy_active', 'hour', 'weekday', 'month', 'total_power

```python
Overall data time range:  2024-01-07 23:00:02.287000 2024-12-31 22:51:01.931000

Test data time range:  2024-10-20 12:15:18.218000 2024-12-31 22:51:01.931000

(Train + Valid) data range:  2024-01-07 23:00:02.287000 2024-10-20 12:15:16.553000
Split 1:
  Train window: 2024-01-07 23:00:02.287000 to 2024-02-03 23:00:01.625324222
  Valid window: 2024-02-03 23:00:10.155160515 to 2024-03-02 22:59:59.675462482

Split 2:
  Train window: 2024-01-07 23:00:02.287000 to 2024-03-02 22:59:59.675462482
  Valid window: 2024-03-02 23:00:08.205298776 to 2024-03-30 22:59:57.725600743

Split 3:
  Train window: 2024-01-07 23:00:02.287000 to 2024-03-30 22:59:57.725600743
  Valid window: 2024-03-30 23:00:06.255437036 to 2024-04-27 22:59:55.775739004

Split 4:
  Train window: 2024-01-07 23:00:02.287000 to 2024-04-27 22:59:55.775739004
  Valid window: 2024-04-27 23:00:04.305575296 to 2024-05-25 22:59:53.825877264

Split 5:
  Train window: 2024-01-07 23:00:02.287000 to 2024-05-25 22:59:53.825877264
  Valid window: 2024-05-25 23:00:02.355713558 to 2024-06-22 23:00:00.405851818

Split 6:
  Train window: 2024-01-07 23:00:02.287000 to 2024-06-22 23:00:00.405851818
  Valid window: 2024-06-22 23:00:08.935688112 to 2024-07-20 22:59:58.455990078

Split 7:
  Train window: 2024-01-07 23:00:02.287000 to 2024-07-20 22:59:58.455990078
  Valid window: 2024-07-20 23:00:06.985826372 to 2024-08-17 22:59:56.506128340

Split 8:
  Train window: 2024-01-07 23:00:02.287000 to 2024-08-17 22:59:56.506128340
  Valid window: 2024-08-17 23:00:05.035964632 to 2024-09-14 22:59:54.556266600

Split 9:
  Train window: 2024-01-07 23:00:02.287000 to 2024-09-14 22:59:54.556266600
  Valid window: 2024-09-14 23:00:03.086102892 to 2024-10-12 23:00:01.136241156

Split 10:
  Train window: 2024-01-07 23:00:02.287000 to 2024-10-12 23:00:01.136241156
  Valid window: 2024-10-12 23:00:09.666077448 to 2024-10-20 12:15:16.553000
```

In [ ]:
print_memory_stats(train, valid, test)

Memory usage:
  Train: 0.00 MB
  Valid: 0.00 MB
  Test:  0.00 MB
  Total: 0.00 MB


## Additional data save if `save_intermediate` is `False` in `get_data_splits`

In [ ]:
# Save data in optimized NumPy format
train_path, valid_path, test_path = save_dataset(
    train=train,
    valid=valid,
    test=test,
    data_dir=data_dir,
    prefix='5_min_wfv',  # Custom prefix for the files
    use_numpy=True  # Use NumPy's compressed format
)

print(f"Data saved to:\n{train_path}\n{valid_path}\n{test_path}")

# Print memory usage statistics
print_memory_stats(train, valid, test)

## Load data code

In [ ]:
# Measure loading time
start_time = time.time()

# Load data using the optimized function
# First try the optimized version, if not available fall back to regular version
train, valid, test = load_dataset(data_dir=data_dir, prefix='8_dev_5_min_wfv', use_numpy=True)

loading_time = time.time() - start_time
print(f"Data loaded in {loading_time/60:.2f} minutes")

# `first_batch` splits

In [ ]:
# If you're using Colab, connect to google drive

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
lag_args = {
    'lag_15min':'15min',
    'lag_1h':'1H',
    'lag_24h':'24H',
    }
#data_dir = "/content/drive/My Drive/extracted_files/"
data_dir = '../data'

columns_to_add_lag = [
            'L1_act_power', 'L1_app_power', 'L1_current', 'L1_react_power', 'L1_voltage',
            'L2_act_power', 'L2_app_power', 'L2_current', 'L2_react_power', 'L2_voltage', 
            'L3_act_power', 'L3_app_power', 'L3_current', 'L3_react_power', 'L3_voltage', 
            'grid_freq', 'powerfactor1', 'powerfactor2', 'powerfactor3'
        ]
device_list = ['df_TRB145-4', 
               #'df_TRB145-5',
               ]

# target
target_feature = 'L2_act_power'

In [4]:
train, valid, test = dataloader.get_train_test_sets(
    data_dir=data_dir,
    lag_freqs=lag_args,
    columns_to_add_lag=columns_to_add_lag,
    target_feature=target_feature,
    device_list=device_list,
    walk_forward_sets=True,
    train_days=28,
    valid_days=28,
    optimize_memory=True,  # Enable memory optimization
    save=True,  # Save processed data
    intermediate_prefix='dev_4_15m_wfv',  # Prefix for saved files
    time_agg='15min',  # Time aggregation frequency
    new_data='first_batch',
    fill_missing=True
)

[INFO] Aggregating data by: 15min
input_features:  ['time', 'device_id', 'hour', 'weekday', 'month', 'L1_act_power_lag_15min', 'L1_act_power_lag_1h', 'L1_act_power_lag_24h', 'L1_app_power_lag_15min', 'L1_app_power_lag_1h', 'L1_app_power_lag_24h', 'L1_current_lag_15min', 'L1_current_lag_1h', 'L1_current_lag_24h', 'L1_react_power_lag_15min', 'L1_react_power_lag_1h', 'L1_react_power_lag_24h', 'L1_voltage_lag_15min', 'L1_voltage_lag_1h', 'L1_voltage_lag_24h', 'L2_act_power_lag_15min', 'L2_act_power_lag_1h', 'L2_act_power_lag_24h', 'L2_app_power_lag_15min', 'L2_app_power_lag_1h', 'L2_app_power_lag_24h', 'L2_current_lag_15min', 'L2_current_lag_1h', 'L2_current_lag_24h', 'L2_react_power_lag_15min', 'L2_react_power_lag_1h', 'L2_react_power_lag_24h', 'L2_voltage_lag_15min', 'L2_voltage_lag_1h', 'L2_voltage_lag_24h', 'L3_act_power_lag_15min', 'L3_act_power_lag_1h', 'L3_act_power_lag_24h', 'L3_app_power_lag_15min', 'L3_app_power_lag_1h', 'L3_app_power_lag_24h', 'L3_current_lag_15min', 'L3_current

# Playground
Here I was just double-checking things and playing around with the data. You can easily omit everything after this part of the notebook.

In [14]:
from pandas import Timestamp

start = Timestamp('2022-10-18 12:00:00+02:00') #  to 
end = Timestamp('2022-11-15 13:45:00+01:00')
diff = end - start
print(f"Difference: {diff}")

Difference: 28 days 02:45:00


In [15]:
# Concatenate all valid splits
valid_arrays = [split[0] for split in valid]
valid_concat = np.concatenate(valid_arrays, axis=0)

# Extract the 'time' column (assumed to be the first column)
all_times = pd.Series(valid_concat[:, 0])

# Convert to pandas datetime if not already
all_times = pd.to_datetime(all_times)

# Create a complete time index with 15min frequency
full_range = pd.date_range(start=all_times.min(), end=all_times.max(), freq='15min', tz=all_times.dt.tz)

# Find missing datetimes
missing_times = full_range.difference(all_times)

print(f"Total timestamps in full range: {len(full_range)}")
print(f"Number of missing timestamps: {len(missing_times)}")
print(missing_times)

Total timestamps in full range: 16675
Number of missing timestamps: 12
DatetimeIndex(['2022-10-30 02:00:00+02:00', '2022-10-30 02:15:00+02:00',
               '2022-10-30 02:30:00+02:00', '2022-10-30 02:45:00+02:00',
               '2022-10-30 02:00:00+01:00', '2022-10-30 02:15:00+01:00',
               '2022-10-30 02:30:00+01:00', '2022-10-30 02:45:00+01:00',
               '2022-10-30 03:00:00+01:00', '2022-10-30 03:15:00+01:00',
               '2022-10-30 03:30:00+01:00', '2022-10-30 03:45:00+01:00'],
              dtype='datetime64[ns, Europe/Prague]', freq='15min')


## Example model training using the loaded data

In [ ]:
grace_period = 0
split_confidence = 0.1

Fodt_5 = fastodt.FastODT(
        depth=20,
        grace_period=grace_period,
        split_confidence=split_confidence,
        option='cb'
    )

In [ ]:
# Use the first split of the training data
if isinstance(train, list):
    # For walk-forward validation structure
    X, y = train[0]

    # Filter for a specific device if needed
    mask = X[:, 1] == "231e45ef-ca7a-4f80-be11-ac7e2da0274c"
    X_device = X[mask]
    y_device = y[mask]

    # Train on a subset
    n_samples = min(200, len(X_device))
    Fodt_5.partial_fit(X_device[:n_samples, 2:], y_device[:n_samples])
else:
    # For single tuple structure
    X, y = train
    mask = X[:, 1] == "231e45ef-ca7a-4f80-be11-ac7e2da0274c"
    X_device = X[mask]
    y_device = y[mask]

    # Train on a subset
    n_samples = min(200, len(X_device))
    Fodt_5.partial_fit(X_device[:n_samples, 2:], y_device[:n_samples])

In [ ]:
# Generate predictions
Y_pred = np.array([Fodt_5.predict(x) for x in X_device[:n_samples, 2:]])